## Projeto 1: Microdados do Enem 2025
### Análise incial dos arquivos
O objetivo é analisar o conteúdo dos arquivos e tirar algumas informações sobre sua qualidade e entender se será necessário realizar algum tipo de tratamento e/ou limpeza dos dados.

### Arquivo 1: ITENS_PROVA_2025.CSV


A partir do dicionário de dados é possível obter informações relevantes:
* As colunas podem ser agrupadas em três grupos: **dados das provas aplicadas**, **dados dos itens de prova**, **dados de TRI dos itens de prova**;
* As colunas ```IN_ITEM_ABAN``` e ```TX_MOTIVO_ABAN``` permitem identificar os itens abandonados e seus motivos. Onde a primeira indica se o item foi abandonado através dos valores 0 (não) e 1 (sim), e a segunda uma breve descrição do motivo.

##### Importando o arquivo e lendo as primeiras 5 linhas.

In [1]:
import pandas as pd
df = pd.read_csv("dados/DADOS/ITENS_PROVA_2025.csv", sep=";", encoding="iso-8859-1")

# Observando as primeiras colunas do conjunto dos dados
df.head()

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO
0,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,BRANCA,1462,0.0,0
1,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1463,0.0,0
2,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1464,0.0,0
3,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1465,0.0,1
4,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1466,0.0,1


##### Analisando os metadados do arquivo

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6105 entries, 0 to 6104
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CO_POSICAO        6105 non-null   int64  
 1   SG_AREA           6105 non-null   str    
 2   CO_ITEM           6105 non-null   int64  
 3   TX_GABARITO       6105 non-null   str    
 4   CO_HABILIDADE     6105 non-null   int64  
 5   IN_ITEM_ABAN      6105 non-null   int64  
 6   TX_MOTIVO_ABAN    75 non-null     str    
 7   NU_PARAM_A        6030 non-null   float64
 8   NU_PARAM_B        6030 non-null   float64
 9   NU_PARAM_C        6030 non-null   float64
 10  TX_COR            6105 non-null   str    
 11  CO_PROVA          6105 non-null   int64  
 12  TP_LINGUA         330 non-null    float64
 13  IN_ITEM_ADAPTADO  6105 non-null   int64  
dtypes: float64(4), int64(6), str(4)
memory usage: 667.9 KB


Com base no resultado obtido é possível tirar algumas conclusões.
* O arquivo possui 14 colunas e 6105 linhas;
* A coluna ```TX_MOTIVO_ABAN``` possui 75 registros não nulos, o que pode indicar que 75 itens foram abandonados
* A coluna ```TP_LINGA``` possui 330 registros não nulos, isso pode estar relacionado com o área de conhecimento da prova.

##### Analisando os itens abandonados e os itens de linguagem estrangeira


Primeiro será observado os campos que chamaram atenção: os itens abandonados e a classificação de lingua estrangeira.

In [3]:
# Criando uma coluna que traduz os códigos da coluna IN_ITEM_ABAN para facilitar a leitura
df["TX_ITEM_ABAN"] = df["IN_ITEM_ABAN"].map({0: "Não", 1: "Sim"})

# Apresenta um resumo da quantidade de itens abandonados
df[["TX_ITEM_ABAN"]].groupby(by="TX_ITEM_ABAN").value_counts().reset_index()

,TX_ITEM_ABAN,count
0,Não,6030
1,Sim,75


O resultado acima no responde duas coisas. Primeiro, a quantidade de itens abandonados corresponde com a quantidade de registros com o motivo preenchido. Segundo, a quantidade de itens NÃO abandonados corresponde com a quantidade de registros preenchidos para os parâmetros A, B e C utilizados no modelo TRI.

In [4]:
# Resumo com a quantidade de itens de lingua estrangeira por área de conhecimento.
df[["SG_AREA", "TP_LINGUA"]].groupby(by="SG_AREA").value_counts().reset_index()

,SG_AREA,TP_LINGUA,count
0,LC,0.0,165
1,LC,1.0,165


O ```TP_LINGUA``` está relacionado a área de conhecimento LC (Linguagens e Códigos) o que faz total sentido. Isso também significa que os valores nulos se referem a itens das demais áreas.

##### Outros pontos de interesse

In [5]:
# Quantidade total de provas criadas
qtde_provas = df["CO_PROVA"].nunique()
print(f"Quantidade total de provas: {qtde_provas}")

Quantidade total de provas: 132


In [6]:
# Quantidade total de provas por cor do caderno
df[["TX_COR", "CO_PROVA"]].groupby(by=["TX_COR",], as_index=False).nunique()

,TX_COR,CO_PROVA
0,AMARELA,12
1,AZUL,12
2,BRANCA,6
3,CINZA,6
4,LARANJA,60
5,LEITOR TELA,12
6,ROXA,12
7,VERDE,12


Comparando o resultado com as provas disponíveis para download no site do [INEP](https://www.gov.br/inep/pt-br/areas-de-atuacao/avaliacao-e-exames-educacionais/enem/provas-e-gabaritos). Nota-se que a cor ROXA se refere as provas de libras, a LARANJA a braile e ledor, LEITOR TELA fazem parte da cor LARANJA. Por fim, a cores BRANCA e CINZA, parecem fazer parte do mesmo "grupo", pois a BRANCA é aplicada no primeiro dia e a CINZA no segundo dia conforme o calendário de aplicação das provas. 

In [ ]:
# Quantidade de questões por área de conhecimento de todas as provas
sg_area = {"CH": "Ciências Humanas",
           "CN": "Ciências da Natureza",
           "LC": "Linguagens e Códigos",
           "MT": "Matemática"
}

df["TX_SG_AREA"] = df["SG_AREA"].map(sg_area)

df_itens = (df[["TX_COR", "TX_SG_AREA"]]
            .groupby(by=["TX_COR", "TX_SG_AREA"], as_index=False)
            .value_counts()
            .pivot(index=["TX_COR"], columns="TX_SG_AREA", values=["count"])
            )

df_itens

count                                            \
TX_SG_AREA  Ciências Humanas Ciências da Natureza Linguagens e Códigos   
TX_COR                                                                   
AMARELA                135.0                135.0                150.0   
AZUL                   135.0                135.0                150.0   
BRANCA                 135.0                  NaN                150.0   
CINZA                    NaN                135.0                  NaN   
LARANJA                675.0                675.0                750.0   
LEITOR TELA            135.0                135.0                150.0   
ROXA                   135.0                135.0                150.0   
VERDE                  135.0                135.0                150.0   

                        
TX_SG_AREA  Matemática  
TX_COR                  
AMARELA          135.0  
AZUL             135.0  
BRANCA             NaN  
CINZA            135.0  
LARANJA          675.0  
LEITOR TELA      135.0  
ROXA             135.0  
VERDE            135.0

As provas de Linguagens e Códigos possuem mais itens que as demais áreas devido as questões de lingua estrangeira, sendo 5 para inglês e 5 para espanhol.

In [39]:
# Verificando as opções de resposta possíveis na tentativa de encontrar alguma incossistência no gabarito
# Quantidade de respostas por alternativa

df[["TX_GABARITO"]].value_counts()

TX_GABARITO
D              1379
B              1350
C              1231
E              1079
A              1027
X                39
Name: count, dtype: int64

Percebe-se que existem 39 itens marcados com a opção X no gabarito. Conforme o dicionário de dados essas questões foram anuladas.

##### Verificando a existência de duplicidades


Como esse conjunto de dados possui informações relacionadas aos itens das provas é importante verificar se existem dados duplicados, nesse caso o mesmo item dentro de uma mesma prova. Porém antes de seguir com o código algumas considerações precisam ser feitas.

* Todas a provas possuem as mesmas questões e a mesma quantidade
* As cores são usadas como uma forma de alterar a ordem das questões, porém sem alterar seu conteúdo.
* Cada prova possui apenas uma cor e pertence a apenas uma área de conhecimento

In [ ]:
# Verificar se existe uma prova presente em mais de um TX_COR
df_prova = (df[["CO_PROVA", "TX_COR", "SG_AREA"]]
            .groupby(by=["CO_PROVA", "TX_COR"], as_index=False)
            .nunique()
            .pivot(index="CO_PROVA", columns=["TX_COR",], values="SG_AREA")
            .fillna(value=0)
            )

df_prova["TOTAL"] = df_prova.sum(axis=1)

# Verifica se existe provas presentes em mais de uma cor
if len(df_prova[df_prova["TOTAL"] > 1]) == 0:
    print("Não existem provas presente em mais de uma cor")
else:
    df_prova[df_prova["TOTAL"] > 1]

Não existem provas presente em mais de uma cor


In [55]:
# Verificar se existe uma prova em mais de uma área de conhecimento
df_prova = (df[["CO_PROVA", "TX_COR", "SG_AREA"]]
            .groupby(by=["CO_PROVA", "SG_AREA"], as_index=False)
            .nunique()
            .pivot(index="CO_PROVA", columns=["SG_AREA",], values="TX_COR")
            .fillna(value=0)
            )

df_prova["TOTAL"] = df_prova.sum(axis=1)

# Verifica se existe provas presentes em mais de uma área de conhecimento
if len(df_prova[df_prova["TOTAL"] > 1]) == 0:
    print("Não existem provas presente em mais de uma área de conhecimento")
else:
    df_prova[df_prova["TOTAL"] > 1]

Não existem provas presente em mais de uma área de conhecimento


Os dois resultados acima indicam que, em relação as provas, não há duplicidades.

In [56]:
# Vericando as duplicidades dos itens
df["CO_ITEM"].value_counts()

CO_ITEM
153930    13
153945    13
118169    13
140709    13
152688    13
          ..
141519     4
125903     4
154019     3
153385     3
150500     3
Name: count, Length: 565, dtype: int64

In [58]:
# Escolhendo o item 153930 e filtrando no dataframe para verificar o que ocorre
df[df["CO_ITEM"] == 153930].sort_values(by="CO_PROVA")

,CO_POSICAO,SG_AREA,CO_ITEM,TX_GABARITO,CO_HABILIDADE,IN_ITEM_ABAN,TX_MOTIVO_ABAN,NU_PARAM_A,NU_PARAM_B,NU_PARAM_C,TX_COR,CO_PROVA,TP_LINGUA,IN_ITEM_ADAPTADO,TX_ITEM_ABAN,TX_SG_AREA
214,5,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,AZUL,1459,0.0,0,Não,Linguagens e Códigos
452,1,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,AMARELA,1460,0.0,0,Não,Linguagens e Códigos
85,3,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,VERDE,1461,0.0,0,Não,Linguagens e Códigos
0,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,BRANCA,1462,0.0,0,Não,Linguagens e Códigos
1,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1463,0.0,0,Não,Linguagens e Códigos
2,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1464,0.0,0,Não,Linguagens e Códigos
3,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1465,0.0,1,Não,Linguagens e Códigos
4,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,LARANJA,1466,0.0,1,Não,Linguagens e Códigos
7,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,ROXA,1467,0.0,0,Não,Linguagens e Códigos
8,2,LC,153930,A,6,0,NaN,2.28024,0.18554,0.19951,ROXA,1468,0.0,0,Não,Linguagens e Códigos


A tabela acima indica que para um mesmo item existe apenas um código de prova atribuída a ela. O código a seguir visa verificar a veracidade dessa afirmação.

In [ ]:
# Realiza a contagem de registros considerando a combinação de CO_PROVA e CO_ITEM
df_item_unico = (df[["CO_PROVA", "CO_ITEM"]]
                 .groupby(by=["CO_PROVA", "CO_ITEM"])
                 .value_counts()
                 .reset_index()
                 )

df_item_unico

,CO_PROVA,CO_ITEM,count
0,1447,55300,1
1,1447,55581,1
2,1447,56054,1
3,1447,56368,1
4,1447,83900,1
...,...,...,...
6100,1582,141616,1
6101,1582,141638,1
6102,1582,141644,1
6103,1582,141746,1


In [62]:
# Filtra apenas os casos acima de 1 na contagem e caso não haja uma mensagem é retornada.

df_item_unico = df_item_unico[df_item_unico["count"] > 1]

if len(df_item_unico) == 0:
    print("Não existem item duplicados para uma mesma prova.")
else:
    df_item_unico


Não existem item duplicados para uma mesma prova.


##### Conclusão

Com base nas análises realizadas no arquivo ITENS_PROVA_2025.csv, foi possível determinar que os estão estão pronto para uso e não há necessidade de realizar nenhum tipo de limpeza ou tratamento de duplicidades nos dados. Além disso, o trabalho possibilitou ampliar os conhecimentos a respeito das provas: o que as compõem, como estão organizadas, como cada prova se relaciona com as questões e vice-versa.